In [3]:
import pandas as pd # main data tool (like Excel in Python)
import numpy as np # math & number operations
import matplotlib.pyplot as plt # basic charts
import seaborn as sns # nicer looking charts
import warnings
warnings.filterwarnings('ignore') # hide unnecessary warning messages
pd.set_option('display.max_columns', None) # show ALL columns
print('All libraries imported successfully!')

All libraries imported successfully!


In [4]:
df = pd.read_csv('D:\Technocolabs\Delivery Five Cities Datasets\delivery_hz.csv') # reads the file
print('File loaded!')
print('Number of rows :', df.shape[0]) # how many records
print('Number of columns :', df.shape[1]) # how many field

File loaded!
Number of rows : 1861600
Number of columns : 17


In [23]:
df.head()

,order_id,region_id,city,courier_id,lng,lat,aoi_id,aoi_type,accept_time,accept_gps_time,accept_gps_lng,accept_gps_lat,delivery_time,delivery_gps_time,delivery_gps_lng,delivery_gps_lat,ds,gps_synced_accept,gps_synced_delivery,accept_to_delivery_min,accept_hour,accept_day,accept_month,time_of_day,gps_drift
0,583722,0,Hangzhou,175,120.17895,30.26401,749,1,2023-10-30 09:20:00,2023-10-30 09:20:00,120.20600,30.28657,2023-10-30 10:30:00,2023-10-30 10:30:00,120.17908,30.26392,1030,True,True,70.0,9,30,10,Morning,0.035181
1,2819059,0,Hangzhou,175,120.17899,30.26336,749,1,2023-10-31 09:47:00,2023-10-31 09:47:00,120.20591,30.28651,2023-10-31 10:40:00,2023-10-31 10:40:00,120.17884,30.26360,1031,True,True,53.0,9,31,10,Morning,0.035463
2,2879432,0,Hangzhou,175,120.17896,30.26404,749,1,2023-10-22 10:11:00,2023-10-22 10:11:00,120.20598,30.28668,2023-10-22 15:03:00,2023-10-22 15:03:00,120.17939,30.26395,1022,True,True,292.0,10,22,10,Morning,0.034981
3,392295,0,Hangzhou,175,120.17897,30.26408,749,1,2023-10-26 09:41:00,2023-10-26 09:41:00,120.20600,30.28657,2023-10-26 10:30:00,2023-10-26 10:30:00,120.17925,30.26465,1026,True,True,49.0,9,26,10,Morning,0.034584
4,231864,0,Hangzhou,175,120.17888,30.26406,749,1,2023-10-31 15:58:00,2023-10-31 15:58:00,120.20605,30.28666,2023-10-31 16:41:00,2023-10-31 16:41:00,120.17886,30.26402,1031,True,True,43.0,15,31,10,Afternoon,0.035382


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1861600 entries, 0 to 1861599
Data columns (total 17 columns):
 #   Column             Dtype  
---  ------             -----  
 0   order_id           int64  
 1   region_id          int64  
 2   city               object 
 3   courier_id         int64  
 4   lng                float64
 5   lat                float64
 6   aoi_id             int64  
 7   aoi_type           int64  
 8   accept_time        object 
 9   accept_gps_time    object 
 10  accept_gps_lng     float64
 11  accept_gps_lat     float64
 12  delivery_time      object 
 13  delivery_gps_time  object 
 14  delivery_gps_lng   float64
 15  delivery_gps_lat   float64
 16  ds                 int64  
dtypes: float64(6), int64(6), object(5)
memory usage: 241.4+ MB


In [6]:
df.isnull().sum()

order_id             0
region_id            0
city                 0
courier_id           0
lng                  0
lat                  0
aoi_id               0
aoi_type             0
accept_time          0
accept_gps_time      0
accept_gps_lng       0
accept_gps_lat       0
delivery_time        0
delivery_gps_time    0
delivery_gps_lng     0
delivery_gps_lat     0
ds                   0
dtype: int64

In [7]:
missing_pct = (df.isnull().sum() / len(df)) * 100
print(missing_pct.sort_values(ascending=False))

order_id             0.0
accept_gps_time      0.0
delivery_gps_lat     0.0
delivery_gps_lng     0.0
delivery_gps_time    0.0
delivery_time        0.0
accept_gps_lat       0.0
accept_gps_lng       0.0
accept_time          0.0
region_id            0.0
aoi_type             0.0
aoi_id               0.0
lat                  0.0
lng                  0.0
courier_id           0.0
city                 0.0
ds                   0.0
dtype: float64


In [8]:
df.isnull().sum().sum() == 0, 'Unexpected missing values found!'
print('Confirmed: No missing values in delivery_cq dataset.')

Confirmed: No missing values in delivery_cq dataset.


In [10]:
df['gps_synced_accept'] = df['accept_time'] == df['accept_gps_time']
df['gps_synced_delivery'] = df['delivery_time'] == df['delivery_gps_time']

print('GPS synced at accept :', df['gps_synced_accept'].sum())
print('GPS synced at delivery :', df['gps_synced_delivery'].sum())

GPS synced at accept : 1861600
GPS synced at delivery : 1861600


In [11]:
print(type(df['accept_time'][0])) # will show <class 'str'> = text
print(df['accept_time'][0]) # shows: 10-22 10:26:00

<class 'str'>
10-30 09:20:00


In [12]:
time_cols = ['accept_time', 'accept_gps_time',
'delivery_time', 'delivery_gps_time']
for col in time_cols:
    df[col] = pd.to_datetime('2023-' + df[col],
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce') # bad values → NaT

In [13]:
print('After conversion:')
print(df[time_cols].dtypes)

After conversion:
accept_time          datetime64[ns]
accept_gps_time      datetime64[ns]
delivery_time        datetime64[ns]
delivery_gps_time    datetime64[ns]
dtype: object


In [14]:
# Example: How many minutes from accept to delivery?
df['accept_to_delivery_min'] = (
(df['delivery_time'] - df['accept_time']).dt.total_seconds() / 60)

In [15]:
print(df['accept_to_delivery_min'].describe().round(1))

count    1861600.0
mean         174.4
std          722.1
min            0.0
25%           61.0
50%          115.0
75%          197.0
max       166499.0
Name: accept_to_delivery_min, dtype: float64


In [16]:
duplicates = df.duplicated(subset=['order_id'])
print('Number of duplicate rows:', duplicates.sum())

Number of duplicate rows: 0


In [17]:
before = len(df)
df = df.drop_duplicates(subset=['order_id'], keep='first')
df = df.reset_index(drop=True) # re-number the rows from 0
after = len(df)
print(f'Rows before: {before:,}')
print(f'Rows after : {after:,}')
print(f'Duplicates removed: {before - after:,}')

Rows before: 1,861,600
Rows after : 1,861,600
Duplicates removed: 0


In [18]:
col = 'accept_to_delivery_min'
Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = (df[col] < lower) | (df[col] > upper)
print(f'Outlier rows found: {outliers.sum():,}')
print(f'Valid range: {lower:.0f} to {upper:.0f} minutes')

Outlier rows found: 103,040
Valid range: -143 to 401 minutes


In [ ]:
df = df[df['accept_to_delivery_min'] >= 0].reset_index(drop=True)
print('Rows after removing negatives:', len(df))

In [19]:
print('Number of rows :', df.shape[0]) # how many records
print('Number of columns :', df.shape[1]) # how many field

Number of rows : 1861600
Number of columns : 20


In [20]:
df['accept_to_delivery_min'] = df['accept_to_delivery_min'].clip(upper=upper)
print('Max duration after clipping:', df['accept_to_delivery_min'].max())


Max duration after clipping: 401.0


In [21]:
# Cell 10a — Extract hour, day, month from accept_time
df['accept_hour'] = df['accept_time'].dt.hour # 0 to 23
df['accept_day'] = df['accept_time'].dt.day # 1 to 31
df['accept_month'] = df['accept_time'].dt.month # 1 to 12
print(df[['accept_time','accept_hour','accept_day','accept_month']].head(3))
# Cell 10b — Group hour into time-of-day buckets
def time_of_day(hour):
    if 0 <= hour < 6: return 'Night'
    elif 6 <= hour < 12: return 'Morning'
    elif 12 <= hour < 17: return 'Afternoon'
    elif 17 <= hour < 21: return 'Evening'
    else: return 'Night'
df['time_of_day'] = df['accept_hour'].apply(time_of_day)
print(df['time_of_day'].value_counts())

          accept_time  accept_hour  accept_day  accept_month
0 2023-10-30 09:20:00            9          30            10
1 2023-10-31 09:47:00            9          31            10
2 2023-10-22 10:11:00           10          22            10
time_of_day
Morning      1169821
Afternoon     554661
Evening       133806
Night           3312
Name: count, dtype: int64


In [28]:
df['gps_drift'] = ((
(df['delivery_gps_lng'] - df['accept_gps_lng'])**2 +
(df['delivery_gps_lat'] - df['accept_gps_lat'])**2
) ** 0.5)
print(df['gps_drift'].describe().round(4))

count    1.861600e+06
mean     3.110000e-02
std      6.316000e-01
min      0.000000e+00
25%      1.050000e-02
50%      1.780000e-02
75%      3.080000e-02
max      1.241112e+02
Name: gps_drift, dtype: float64


In [29]:
df['delivery_speed_proxy'] = df['gps_drift'] / (df['accept_to_delivery_min'] + 1)
print('Speed proxy stats:')
print(df['delivery_speed_proxy'].describe().round(4))

Speed proxy stats:
count    1.861600e+06
mean     5.000000e-04
std      1.760000e-02
min      0.000000e+00
25%      1.000000e-04
50%      2.000000e-04
75%      3.000000e-04
max      1.768570e+01
Name: delivery_speed_proxy, dtype: float64


In [30]:
df['delivery_month'] = df['delivery_time'].dt.month
print(df['delivery_month'].value_counts().sort_index())

delivery_month
5     232795
6     297648
7     268748
8     314858
9     348814
10    398438
11       294
12         5
Name: count, dtype: int64


In [31]:
# Check unique city values (case-sensitive)
print(df['city'].unique())
print(df['city'].value_counts())


['Hangzhou']
city
Hangzhou    1861600
Name: count, dtype: int64


In [32]:
# Find all object (text) columns
text_cols = df.select_dtypes(include='object').columns
print('Text columns:', text_cols.tolist())
# ['city', 'accept_time', 'accept_gps_time', 'delivery_time', 'delivery_gps_time']

# Check each one
for col in text_cols:
    unique_vals = df[col].nunique()
    print(f'{col}: {unique_vals} unique values')

# city               → should be 1 (only "Hangzhou")
# accept_time        → will be many (timestamps)
# delivery_time      → will be many (timestamps)

Text columns: ['city', 'time_of_day']
city: 1 unique values
time_of_day: 4 unique values


In [33]:
print('Final dataset shape:', df.shape)
print()
print('Missing values left:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print('New columns created:')
new_cols = ['gps_synced_accept', 'gps_synced_delivery',
'accept_to_delivery_min', 'accept_hour', 'accept_day',
'accept_month', 'time_of_day', 'gps_drift',
'delivery_speed_proxy', 'delivery_month']
print(new_cols)

Final dataset shape: (1861600, 27)

Missing values left:
Series([], dtype: int64)

New columns created:
['gps_synced_accept', 'gps_synced_delivery', 'accept_to_delivery_min', 'accept_hour', 'accept_day', 'accept_month', 'time_of_day', 'gps_drift', 'delivery_speed_proxy', 'delivery_month']


In [35]:
df.to_csv('delivery_hz_cleaned.csv', index=False)
print('File saved as: delivery_hz_cleaned.csv')
print(f'Total rows saved: {len(df):,}')

File saved as: delivery_hz_cleaned.csv
Total rows saved: 1,861,600
